# Multivariate Statistical Analysis Practice Notebook

Hands-on companion for **07_multivariate_analysis.md**.

Topics:
1. PCA
2. Factor-style latent analysis
3. MANOVA-style multivariate group comparison
4. Canonical Correlation Analysis
5. Cluster analysis
6. Discriminant analysis (LDA)
7. Multidimensional Scaling (MDS)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.cross_decomposition import CCA
from sklearn.cluster import KMeans
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import MDS
from sklearn.metrics import silhouette_score
from scipy import stats
np.random.seed(42)


## Create Multivariate Dataset


In [ ]:
n=180
z1=np.random.normal(size=n)
z2=np.random.normal(size=n)
X=pd.DataFrame({
 'x1':2*z1+0.2*np.random.normal(size=n),
 'x2':1.8*z1+0.3*np.random.normal(size=n),
 'x3':2.2*z2+0.2*np.random.normal(size=n),
 'x4':1.7*z2+0.3*np.random.normal(size=n),
 'x5':z1+z2+0.4*np.random.normal(size=n)
})
groups=np.repeat(['A','B','C'], n//3)
X.head()


## 1. Principal Component Analysis (PCA)


In [ ]:
pca=PCA()
X_pca=pca.fit_transform(X)
evr=pca.explained_variance_ratio_
pd.DataFrame({'PC':[f'PC{i+1}' for i in range(len(evr))],'Explained Variance Ratio':evr})


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(range(1,len(evr)+1), np.cumsum(evr), marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance')
plt.show()


## 2. Factor Analysis


In [ ]:
fa=FactorAnalysis(n_components=2, random_state=42)
F=fa.fit_transform(X)
loadings=pd.DataFrame(fa.components_.T, index=X.columns, columns=['Factor1','Factor2'])
loadings


## 3. MANOVA-style Multivariate Group Comparison

A lightweight practical approximation compares group means across multiple responses using per-feature ANOVA, useful for intuition before full MANOVA.


In [ ]:
Xg=X.copy()
Xg['group']=groups
anova_rows=[]
for col in X.columns:
    a=Xg.loc[Xg.group=='A',col]
    b=Xg.loc[Xg.group=='B',col]
    c=Xg.loc[Xg.group=='C',col]
    f,p=stats.f_oneway(a,b,c)
    anova_rows.append((col,f,p))
pd.DataFrame(anova_rows, columns=['Variable','F','p_value'])


## 4. Canonical Correlation Analysis (CCA)


In [ ]:
X_block=X[['x1','x2','x5']]
Y_block=X[['x3','x4']]
cca=CCA(n_components=1)
U,V=cca.fit_transform(X_block, Y_block)
cca_corr=np.corrcoef(U[:,0],V[:,0])[0,1]
cca_corr


## 5. Cluster Analysis (k-means)


In [ ]:
kmeans=KMeans(n_clusters=3, random_state=42, n_init=10)
labels=kmeans.fit_predict(X)
sil=silhouette_score(X, labels)
pd.DataFrame({'Cluster':labels}).value_counts().rename('Count').reset_index(), sil


Visualize clusters on first two PCs.


In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('k-means clusters in PCA space')
plt.show()


## 6. Discriminant Analysis (LDA)


In [ ]:
lda=LinearDiscriminantAnalysis(n_components=2)
X_lda=lda.fit_transform(X, groups)
plt.figure(figsize=(6,5))
plt.scatter(X_lda[:,0], X_lda[:,1], c=pd.Categorical(groups).codes)
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.title('LDA projection by known groups')
plt.show()


## 7. Multidimensional Scaling (MDS)


In [ ]:
mds=MDS(n_components=2, random_state=42)
X_mds=mds.fit_transform(X)
plt.figure(figsize=(6,5))
plt.scatter(X_mds[:,0], X_mds[:,1], c=labels)
plt.xlabel('MDS1')
plt.ylabel('MDS2')
plt.title('MDS map')
plt.show()


## 8. Summary Table


In [ ]:
summary=pd.DataFrame({
 'Measure':['PCA EVR PC1','PCA EVR PC2','CCA corr','Silhouette score'],
 'Value':[evr[0], evr[1], cca_corr, sil]
})
summary


## 9. Mini Exercises
1. Change PCA retained components and inspect variance.
2. Try 2 vs 4 clusters.
3. Replace synthetic data with your 59-feature structural dataset.
4. Compare PCA and FactorAnalysis latent spaces.
5. Use CCA on input vs output blocks from surrogate data.
